# Prescribed BVOC emissions, all 19 MEGAN-fed species: CTRL_PD vs CTRL_FUT
Forcing files of the all-species fBVOC runs (2001–2014 climatologies, `run_bvoc_climatology_array.sh`):

| Source | Files | Used by |
|---|---|---|
| CTRL_PD | `ems_<CTRL_PD>_2001-2014_<VAR>.nc` (year 2000) | 14c `LCC_PD_fBVOCctrlPD`; same values relabelled to 2100 in 24c `LCC_FUT_fBVOCctrlPD` |
| CTRL_FUT | `ems_<CTRL_FUT>_2001-2014_on_yr2100_<VAR>.nc` | 24d `LCC_FUT_fBVOCctrlFUT` |

The `_on_yr2100` files differ from the year-2000 files only in their dates, so columns compare **sources**, not years.
ISOP and MTERP come from `SF*` (MEGAN + biomass burning); the other 17 from `MEG_*` (MEGAN only). 8 species are half-hourly, 11 monthly (`MONTHLY_VARS`).
These maps inspect the forcing files; they do not validate the fluxes applied by CAM. The ISOP/MTERP-only version of this notebook is in `prescribe-just-ISOP-MTERP/`.

In [ ]:
from pathlib import Path
import calendar, sys
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.path as mpath
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point
from matplotlib.colors import TwoSlopeNorm
from IPython.display import display

sys.path.insert(0, str(Path.cwd()))   # species list, molar masses, names from the emission script
from create_bvoc_emissions_climatology_staged import MOLAR_MASS, MONTHLY_VARS, OUTPUT_VAR_NAME, AVOGADRO

DATA_DIR = Path("/cluster/shared/noresm/inputdata/atm/cam/prescribed_data/bvoc_emissions")
# Keep up to date with the CTRL versions (CLAUDE.md "When a run version changes").
SOURCES = {"CTRL_PD": "ems_NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-20261023_2001-2014",
           "CTRL_FUT": "ems_NF2100ssp585norbc_tropstratchem_nudg_ctrl_f19_f19-20261023_2001-2014_on_yr2100"}
SPECIES = list(MOLAR_MASS)            # history-field names: SFISOP, SFMTERP, MEG_BCARY, ...
MASK_FILE = Path("/cluster/home/adelez/BOREAL-FOREST-EXPANSION/diagnostics/check-simulation-equilibrium/lcc-modified-cells/modified_cells.nc")
MDAYS = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], float)
SEASONS = {"DJF": (12, 1, 2), "MAM": (3, 4, 5), "JJA": (6, 7, 8), "SON": (9, 10, 11)}
CHUNK = 48 * 7                        # half-hourly records read per step (one week)

## 1. One pass per file
Annual and seasonal mean maps, and area-weighted series (global, north of 45°N, modified cells). Half-hourly files are read one week at a time; their annual mean is the mean of all 17520 records. Monthly files are averaged with month-length weights. Seasonal means of half-hourly files weight every record equally, so they are also day-weighted.

In [ ]:
with xr.open_dataset(MASK_FILE) as ds:
    mask = ds["modified"].load().astype(bool)
first = xr.open_dataset(DATA_DIR / f"{SOURCES['CTRL_PD']}_SFISOP.nc", decode_times=False)
lat, lon = first.lat, first.lon
for name, a, b in (("lat", mask.lat.values, lat.values), ("lon", mask.lon.values % 360, lon.values % 360)):
    if a.shape != b.shape or np.abs(a - b).max() > 1e-5:   # require the same grid; never remap silently
        raise ValueError(f"mask and emission {name} differ")
mask = mask.assign_coords(lat=lat, lon=lon)
coslat = np.cos(np.deg2rad(lat)) * xr.ones_like(lon)
AREA = coslat / float(coslat.sum()) * 4 * np.pi * 6.371e6**2          # m2 per cell
REGIONS = {"global": coslat, ">45N": coslat.where(lat > 45, 0), "modified": coslat.where(mask, 0)}

def to_tg_per_yr(field, var, weights):
    """molecules/cm2/s -> Tg(species)/yr over the cells with nonzero weight."""
    kg = field / (AVOGADRO / MOLAR_MASS[var] * 1e-4)
    return float((kg * AREA).where(weights > 0, 0).sum()) * 365 * 86400 / 1e9

annual, seasonal, series, units = {}, {}, {}, {}
for var in SPECIES:
    name, monthly = OUTPUT_VAR_NAME[var], var in MONTHLY_VARS
    for src, prefix in SOURCES.items():
        path = DATA_DIR / f"{prefix}_{var}.nc"
        print(f"Reading {path.name}", flush=True)
        with xr.open_dataset(path, decode_times=False) as ds:
            units[var] = ds[name].attrs.get("units", "?")
            dates = ds["date"].values.astype(int)
            months = (dates // 100) % 100
            if monthly:
                x = ds[name].load()
                annual[var, src] = x.weighted(xr.DataArray(MDAYS, dims="time")).mean("time")
                seasonal[var, src] = {s: x.isel(time=np.flatnonzero(np.isin(months, m)))
                                       .weighted(xr.DataArray(MDAYS[np.array(m) - 1], dims="time")).mean("time")
                                      for s, m in SEASONS.items()}
                xpos = np.arange(1, 13) + 0.5                      # mid-month on the month axis
                series[var, src] = {r: pd.Series(x.weighted(w).mean(("lat", "lon")).values, index=xpos)
                                    for r, w in REGIONS.items()}
            else:
                total, sums, counts, regional = 0, {}, {}, {r: [] for r in REGIONS}
                for start in range(0, len(dates), CHUNK):
                    x = ds[name].isel(time=slice(start, start + CHUNK)).load()
                    total = total + x.sum("time")
                    for s, m in SEASONS.items():
                        idx = np.flatnonzero(np.isin(months[start:start + CHUNK], m))
                        if len(idx):
                            sums[s] = sums.get(s, 0) + x.isel(time=idx).sum("time")
                            counts[s] = counts.get(s, 0) + len(idx)
                    for r, w in REGIONS.items():
                        regional[r].append(x.weighted(w).mean(("lat", "lon")).values)
                annual[var, src] = total / len(dates)
                seasonal[var, src] = {s: sums[s] / counts[s] for s in SEASONS}
                # daily means; dates of 1 Jan of the next year (last record, phase 1800 s) go to the right edge
                xd = ((dates // 100) % 100) + ((dates % 100) - 1) / MDAYS[((dates // 100) % 100) - 1]
                xd = xd + 12 * (dates // 10000 - dates[0] // 10000)
                series[var, src] = {r: pd.Series(np.concatenate(v)).groupby(xd).mean() for r, v in regional.items()}
first.close()

## 2. Totals per species and region (Tg of species per year)
Cross-species comparison in one unit (molecules/cm²/s converted back with the CAM molar mass). *modified* = LCC-modified cells (1161 cells, 46–82°N).

In [ ]:
rows = []
for var in SPECIES:
    row = {"species": OUTPUT_VAR_NAME[var], "field": var, "sampling": "monthly" if var in MONTHLY_VARS else "30 min"}
    for r, w in REGIONS.items():
        pd_, fut = (to_tg_per_yr(annual[var, s], var, w) for s in SOURCES)
        row[f"{r} PD"], row[f"{r} FUT"], row[f"{r} FUT/PD"] = pd_, fut, fut / pd_ if pd_ else np.nan
    rows.append(row)
totals = pd.DataFrame(rows).set_index("species")
with pd.option_context("display.float_format", "{:.3g}".format):
    display(totals)

## 3. Annual-mean maps: CTRL_PD, CTRL_FUT, FUT − PD
One row per species, each with its own colour scale (PD and FUT share it). Split into half-hourly and monthly species.

In [ ]:
def plot_rows(vars_, title, extent=None):
    proj = ccrs.NorthPolarStereo() if extent else ccrs.PlateCarree()
    fig, axes = plt.subplots(len(vars_), 3, figsize=(15 if not extent else 11, 3 * len(vars_)),
                             subplot_kw={"projection": proj}, constrained_layout=True, squeeze=False)
    circle = mpath.Path(np.column_stack([np.sin(t := np.linspace(0, 2 * np.pi, 100)), np.cos(t)]) * 0.5 + 0.5)
    for row, var in zip(axes, vars_):
        pdf, fut = (annual[var, s] for s in SOURCES)
        if extent:
            pdf, fut = (f.where(f.lat > 45, drop=True) for f in (pdf, fut))
        diff = fut - pdf
        vmax = max(float(pdf.max()), float(fut.max()), 1e-30)
        dmax = max(float(abs(diff).max()), 1e-30)
        for ax, field, kw, label in ((row[0], pdf, dict(cmap="viridis", vmin=0, vmax=vmax), "CTRL_PD"),
                                     (row[1], fut, dict(cmap="viridis", vmin=0, vmax=vmax), "CTRL_FUT"),
                                     (row[2], diff, dict(cmap="RdBu_r", norm=TwoSlopeNorm(0, -dmax, dmax)), "FUT − PD")):
            values, lons = add_cyclic_point(field.values, coord=field.lon.values)
            mesh = ax.pcolormesh(lons, field.lat, values, transform=ccrs.PlateCarree(), shading="auto", **kw)
            if extent:
                ax.set_extent(extent, crs=ccrs.PlateCarree()); ax.set_boundary(circle, transform=ax.transAxes)
            else:
                ax.set_global()
            ax.coastlines(linewidth=0.5)
            ax.set_title(f"{OUTPUT_VAR_NAME[var]} — {label}")
            if label != "CTRL_PD":
                fig.colorbar(mesh, ax=ax, shrink=0.8, label=units[var] if label == "CTRL_FUT" else f"Δ {units[var]}")
    fig.suptitle(title)
    plt.show()

HALF_HOURLY = [v for v in SPECIES if v not in MONTHLY_VARS]
MONTHLY = [v for v in SPECIES if v in MONTHLY_VARS]
plot_rows(HALF_HOURLY, "Annual-mean prescribed flux, half-hourly species (2001–2014 climatology)")
plot_rows(MONTHLY, "Annual-mean prescribed flux, monthly species (2001–2014 climatology)")

## 4. North of 45°N (forest-expansion region)
Same layout as section 3, polar stereographic.

In [ ]:
plot_rows(HALF_HOURLY, "Annual-mean prescribed flux north of 45°N, half-hourly species", extent=[-180, 180, 45, 90])
plot_rows(MONTHLY, "Annual-mean prescribed flux north of 45°N, monthly species", extent=[-180, 180, 45, 90])

## 5. Seasonal cycles: global and modified cells
Area-weighted means; daily means for half-hourly species, monthly values (markers) for monthly species. Units per panel.

In [ ]:
COLORS = {"CTRL_PD": "#25738A", "CTRL_FUT": "#D95F0E"}
for region in ("global", "modified"):
    fig, axes = plt.subplots(4, 5, figsize=(20, 13), sharex=True, constrained_layout=True)
    for ax, var in zip(axes.flat, SPECIES):
        for src in SOURCES:
            s = series[var, src][region]
            ax.plot(s.index, s.values, color=COLORS[src], label=src, linewidth=1,
                    marker="o" if var in MONTHLY_VARS else None, markersize=3)
        ax.set_title(f"{OUTPUT_VAR_NAME[var]} ({'monthly' if var in MONTHLY_VARS else '30 min'})")
        ax.set_ylabel(units[var], fontsize=8)
        ax.set_xticks(np.arange(1, 13) + 0.5, [m[0] for m in calendar.month_abbr[1:]])
        ax.grid(alpha=0.2)
    axes.flat[-1].axis("off")
    axes.flat[0].legend()
    fig.suptitle(f"Prescribed flux, area-weighted mean over {region} cells")
    plt.show()

## 6. Seasonal maps north of 45°N (per species, CTRL_PD and CTRL_FUT)
Optional detail: set `SEASONAL_SPECIES` (history-field names) to look at more species.

In [ ]:
SEASONAL_SPECIES = ["SFISOP", "SFMTERP", "MEG_BCARY", "MEG_CH3OH"]
circle = mpath.Path(np.column_stack([np.sin(t := np.linspace(0, 2 * np.pi, 100)), np.cos(t)]) * 0.5 + 0.5)
for var in SEASONAL_SPECIES:
    panels = {(src, s): seasonal[var, src][s].where(lat > 45, drop=True) for src in SOURCES for s in SEASONS}
    vmax = max(max(float(f.max()) for f in panels.values()), 1e-30)
    fig, axes = plt.subplots(2, 4, figsize=(15, 7.5), subplot_kw={"projection": ccrs.NorthPolarStereo()}, constrained_layout=True)
    for ax, ((src, s), field) in zip(axes.flat, panels.items()):
        values, lons = add_cyclic_point(field.values, coord=field.lon.values)
        mesh = ax.pcolormesh(lons, field.lat, values, transform=ccrs.PlateCarree(), cmap="YlGn", vmin=0, vmax=vmax, shading="auto")
        ax.set_extent([-180, 180, 45, 90], crs=ccrs.PlateCarree()); ax.set_boundary(circle, transform=ax.transAxes)
        ax.coastlines(linewidth=0.5); ax.set_title(f"{src} {s}")
    fig.colorbar(mesh, ax=axes.ravel().tolist(), shrink=0.7, label=units[var])
    fig.suptitle(f"{OUTPUT_VAR_NAME[var]} — seasonal mean prescribed flux north of 45°N")
    plt.show()